<a href="https://colab.research.google.com/github/hugoggarciamartin-lab/BET-Engine-INS-GNSS/blob/main/bet_eng_audit.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Audit: BET-Engine (RTS Smoother & Thrust Reconstruction)
Kinematic convergencea analysis, $3σ tolerance bands, and dynamic validation
**Note:** Vectorized tensor execution forced.

In [9]:
import os
if not os.path.exists('/content/BET-Engine-INS-GNSS'):
  !git clone https://github.com/hugoggarciamartin-lab/BET-Engine-INS-GNSS.git

In [9]:
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from scipy.interpolate import interp1d

# Colab mount and path injection
project_root = Path("/content/BET-Engine-INS-GNSS") # Changed to match the default clone directory name
sys.path.append(str(project_root))
sys.path.append(str(project_root / "config"))
sys.path.append(str(project_root / "source" / "phase3_nav_eskf_rts"))

from geodesy_math import calc_radii, geodetic_to_enu, quat2eul
from kinematics_ins import calc_isa_atmosphere, calc_mach_number
from config.config_parser import ConfigParser

plt.rcParams.update({'figure.figsize': (10, 4), 'axes.grid': True})

In [10]:
cfg_parser = ConfigParser(project_root / "config" / "config_baseline.yaml")
params = cfg_parser.parse()

npz_file = project_root / "data" / "results" / "rts_eskf_output_state.npz"
master_csv = project_root / "data" / "aligned_data" / "master_flight_data.csv"
mass_csv = project_root / "data" / "raw" / "flight_mass_prof_data.csv"
cfd_csv = project_root / "data" / "raw" / "cfd_drag_model.csv"

with np.load(npz_file, allow_pickle=True) as data:
  x_nom_full = data['x_nom']
  p_full = data['P']

step = 40
x_nom = x_nom_full[::step]
p = p_full[::step]
time = np.arange(len(x_nom_full))[::step] * (1.0 / 400.0)

lat, lon, alt = x_nom[:, 0], x_nom[:, 1], x_nom[:, 2]

rm, rn = calc_radii(lat)

NameError: name 'ConfigParser' is not defined

In [ ]:
sig_lat_deg = np.rad2deg(np.sqrt(p[:, 0, 0]) / (rm + alt))
sig_lon_deg = np.rad2deg(np.sqrt(p[:, 1, 1]) / ((rn + alt) * np.cos(lat)))
sig_alt_m = np.sqrt(p[:, 2, 2])

pos_vars = [np.rad2deg(lat), np.rad2deg(lon), alt]
pos_sigs = [sig_lat_deg, sig_lon_deg, sig_alt_m]
pos_labels = ["Latitude (deg)", "Longitude (deg)", "Relative Altitude (m)"]

fig, axes = plt.subplots(1, 3, figsize=(18, 4))
for i in range(3):
    axes[i].plot(time, pos_vars[i], "k-", lw=1.2, label="Nominal State (RTS)")
    axes[i].fill_between(
        time, pos_vars[i] - 3 * pos_sigs[i], pos_vars[i] + 3 * pos_sigs[i],
        color="r", alpha=0.2, label="$3\sigma$ Bound"
    )
    axes[i].set_title(pos_labels[i])
    axes[i].set_xlabel("Time (s)")
    axes[i].legend()
plt.tight_layout()
plt.show()

In [ ]:
e, n, u = geodetic_to_enu(lat, lon, alt)

fig = plt.figure(figsize=(10, 8))
ax = fig.add_subplot(111, projection="3d")
ax.plot(e, n, u, color="navy", lw=1.5, label="BET Trajectory (ENU)")
ax.set_xlabel("East (m)")
ax.set_ylabel("North (m)")
ax.set_zlabel("Altitude (m)")
ax.set_title("3D Spatial Reconstruction (Subsampled)")
ax.legend()
plt.show()

In [ ]:
# Data loading (400 Hz)
master_df = pd.read_csv(master_csv)
time_master = master_df["time"].values
f_raw_x = master_df["f_X"].values

df_mass = pd.read_csv(mass_csv)
interp_mass = interp1d(df_mass["time"].values, df_mass["mass_kg"].values, kind="linear", fill_value="extrapolate")
mass_arr = interp_mass(time_master)

df_cfd = pd.read_csv(cfd_csv)
interp_cd = interp1d(df_cfd["Mach"].values, df_cfd["C_D_CFD"].values, kind="linear", fill_value="extrapolate")

# Aislamiento de la fuerza específica
bias_a_x = x_nom_full[:, 10]
f_real_x = f_raw_x - bias_a_x

alt_full = np.abs(x_nom_full[:, 2])
v_n_vec_full = x_nom_full[:, 3:6]
v_tas_full = np.linalg.norm(v_n_vec_full, axis=1)

# Vectorized Execution
t_loc, p_loc = calc_isa_atmosphere(
    alt_full, params["p0_isa"], params["t0_isa"], params["l_isa"], params["g_e"]
)
rho_arr = p_loc / (287.0528 * t_loc)

mach_arr = calc_mach_number(
    v_n_vec_full, alt_full, params["p0_isa"], params["t0_isa"], params["l_isa"], params["g_e"]
)

# Thrust Calculation
cd_sim = interp_cd(mach_arr)
q_dyn = 0.5 * rho_arr * (v_tas_full**2)
drag_sim = q_dyn * params["s_ref_m2"] * cd_sim
# Reconstructed Thrust
thrust_rec = (mass_arr * f_real_x) + drag_sim

# Visualization (Engine On Phase)
idx_burn = time_master <= params.get("time_eng_off", 25.0)

fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(time_master[idx_burn], thrust_rec[idx_burn], "k-", lw=1.5, label="Reconstructed Thrust")
ax.set_title("Engine Thrust Temporal Evolution")
ax.set_ylabel("Thrust (N)")
ax.set_xlabel("Time (s)")
ax.legend()
plt.show()

